In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd().parent  
DATA_RAW = PROJECT_ROOT / "data" / "raw"


print("Project root:", PROJECT_ROOT)
print("Data files:")
for f in sorted(DATA_RAW.glob("*.csv")):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
#train
train_txn = pd.read_csv(DATA_RAW / "train_transaction.csv")
train_id = pd.read_csv(DATA_RAW / "train_identity.csv")

print("train_transaction:", train_txn.shape)
print("train_identity:   ", train_id.shape)

In [ ]:
# Что за колонки в transaction?
print("=" * 60)
print("train_transaction — первые 10 колонок:")
print(train_txn.dtypes.head(10))
print()
print("Последние 10 колонок:")
print(train_txn.dtypes.tail(10))
print()
print("Распределение типов данных:")
print(train_txn.dtypes.value_counts())

In [ ]:
train_txn.groupby("card6")[["isFraud"]].count()

In [ ]:
20663/(569877+20663)

In [ ]:
# Target: насколько редок fraud?
print("Распределение isFraud:")
print(train_txn["isFraud"].value_counts())
print()
print(f"Процент fraud: {train_txn['isFraud'].mean() * 100:.3f}%")
print()

# TransactionDT: диапазон времени
dt_min = train_txn["TransactionDT"].min()
dt_max = train_txn["TransactionDT"].max()
days_span = (dt_max - dt_min) / 86400  # секунды -> дни
print(f"TransactionDT: min={dt_min}, max={dt_max}")
print(f"Временной диапазон: {days_span:.1f} дней")

In [ ]:
# Первые 5 транзакций (только ключевые колонки, чтобы не разъехалось)
key_cols = ["TransactionID", "isFraud", "TransactionDT", "TransactionAmt",
            "ProductCD", "card1", "card4", "card6", "P_emaildomain"]
train_txn[key_cols].head()

In [ ]:
# Теперь identity — что туда записано?
print("Колонки train_identity:")
print(train_id.columns.tolist())
print()

# Первые 3 строки
train_id.head(3)

In [ ]:
# Список колонок
print("Колонки identity:")
print(train_id.columns.tolist())
print()
print(f"Всего колонок: {len(train_id.columns)}")
print()

# Первые 3 строки в транспонированном виде — удобнее читать
print("Первые 3 строки (в транспонированном виде):")
train_id.head(3).T

In [ ]:
# Память: сколько весит датафрейм в RAM (а не на диске)
def memory_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print(f"train_transaction: {memory_mb(train_txn):.0f} MB in RAM")
print(f"train_identity:    {memory_mb(train_id):.0f} MB in RAM")
print(f"Итого:             {memory_mb(train_txn) + memory_mb(train_id):.0f} MB in RAM")

In [ ]:
def reduce_mem_usage(df, verbose=True):
    """
    Итеративно проходит по колонкам dataframe и понижает их тип
    до минимально необходимого. Не трогает object-колонки (строки).
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == object:
            continue  # строки не трогаем

        c_min = df[col].min()
        c_max = df[col].max()

        if str(col_type).startswith("int"):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        else:
            # float
            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"Память: {start_mem:.0f} MB -> {end_mem:.0f} MB "
              f"(уменьшено на {reduction:.1f}%)")
    return df

In [ ]:
print("Оптимизация train_transaction...")
train_txn = reduce_mem_usage(train_txn)
print()
print("Оптимизация train_identity...")
train_id = reduce_mem_usage(train_id)

In [ ]:
def optimize_categoricals(df, verbose=True):
    """
    Конвертирует object-колонки в category.
    Категории эффективнее, когда уникальных значений много меньше, чем строк.
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    object_cols = df.select_dtypes(include="object").columns
    for col in object_cols:
        n_unique = df[col].nunique()
        n_total = len(df[col])
        # Конвертируем, если уникальных меньше 50% от всех значений
        if n_unique / n_total < 0.5:
            df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"Память: {start_mem:.0f} MB -> {end_mem:.0f} MB "
              f"(уменьшено на {reduction:.1f}%)")
    return df


print("Оптимизация категорий train_transaction...")
train_txn = optimize_categoricals(train_txn)
print()
print("Оптимизация категорий train_identity...")
train_id = optimize_categoricals(train_id)

In [ ]:
# Merge: к каждой транзакции пристёгиваем identity (если есть)
# how="left" — сохраняем все транзакции, identity опциональна
train = train_txn.merge(train_id, on="TransactionID", how="left")

print(f"train: {train.shape}")
print(f"Память: {train.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
print()

# Сколько транзакций получили identity после merge?
has_identity = train["id_01"].notna().sum()
print(f"Транзакций с identity: {has_identity:,} "
      f"({has_identity / len(train) * 100:.1f}%)")

In [ ]:
# Директория для промежуточных данных
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

# Сохраняем train
out_path = DATA_INTERIM / "train.parquet"
train.to_parquet(out_path, engine="pyarrow", compression="snappy")

size_mb = out_path.stat().st_size / 1024**2
print(f"Сохранено: {out_path}")
print(f"Размер файла: {size_mb:.0f} MB")

In [ ]:
import time

t0 = time.time()
train_check = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
elapsed = time.time() - t0

print(f"Загружено за {elapsed:.2f} секунд")
print(f"Shape: {train_check.shape}")
print(f"Память: {train_check.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

# Проверяем, что типы сохранились
print()
print("Типы нескольких колонок:")
print(train_check[["TransactionAmt", "ProductCD", "card1", "DeviceInfo"]].dtypes)

In [ ]:
# Удаляем train из памяти, чтобы не перегружать ноутбук
del train, train_txn, train_id, train_check
import gc
gc.collect()

# Загружаем test — те же две таблицы
print("Загрузка test_transaction.csv...")
test_txn = pd.read_csv(DATA_RAW / "test_transaction.csv")
print("Загрузка test_identity.csv...")
test_id = pd.read_csv(DATA_RAW / "test_identity.csv")

print(f"test_transaction: {test_txn.shape}")
print(f"test_identity:    {test_id.shape}")


In [ ]:
# Оптимизация типов
print("Оптимизация test_transaction...")
test_txn = reduce_mem_usage(test_txn)
print()
print("Оптимизация test_identity...")
test_id = reduce_mem_usage(test_id)
print()

# Оптимизация категорий
print("Категории test_transaction...")
test_txn = optimize_categoricals(test_txn)
print()
print("Категории test_identity...")
test_id = optimize_categoricals(test_id)

In [ ]:
test = test_txn.merge(test_id, on="TransactionID", how="left")

print(f"test: {test.shape}")
print(f"Память: {test.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
print()

# Сохраняем
out_path = DATA_INTERIM / "test.parquet"
test.to_parquet(out_path, engine="pyarrow", compression="snappy")
size_mb = out_path.stat().st_size / 1024**2
print(f"Сохранено: {out_path.name}, размер: {size_mb:.0f} MB")

In [ ]:
# Что у нас в interim?
for f in sorted(DATA_INTERIM.glob("*.parquet")):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size_mb:.1f} MB")